#### Dense Retrieval Demo for RAG
- how to use dense retrieval (neural embeddings) as a retriever in a Retrieval-Augmented Generation (RAG) pipeline, using the same Wikipedia dataset 

- Dense retrieval captures semantic similarity, going beyond exact keyword matches.

**Install and Import Required Libraries**

Install `sentence-transformers`, `faiss-cpu`, `nltk`, and `pandas` if not already installed.

In [1]:
# If running for the first time, uncomment the following lines:
# !pip install pandas sentence-transformers faiss-cpu nltk

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import nltk

from nltk.tokenize import word_tokenize

#nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bhupe\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

**Load the Wikipedia Dataset**

Load the CSV file with Wikipedia articles. 

In [2]:
import pandas as pd

# Path to the Wikipedia CSV file (update if needed)
data_path = r'D:\AI-DATASETS\02-MISC-large\GenAI-LLMs\chromadb\data\vector_database_wikipedia_articles_embedded.csv'
#data_path = '../data/wikipedia_sample.csv'

# Load the dataset
df = pd.read_csv(data_path)

# Show a sample of the data
df.sample(5)

,id,url,title,text,title_vector,content_vector,vector_id
4689,14788,https://simple.wikipedia.org/wiki/Oklahoma,Oklahoma,Oklahoma () is a state that is in the southern...,"[-0.0024560827296227217, -0.021700330078601837...","[0.002701716497540474, -0.010304819792509079, ...",4689
23828,92032,https://simple.wikipedia.org/wiki/Harold%20Shi...,Harold Shipman,Harold Frederick Shipman (14 January 1946 – 13...,"[-0.006556988228112459, -0.003989354241639376,...","[0.0019171856110915542, 0.004875927232205868, ...",23828
5823,18867,https://simple.wikipedia.org/wiki/Satellite%20...,Satellite (artificial),A satellite is an object that orbits another o...,"[-0.009827318601310253, -0.0015308806905522943...","[0.005813933908939362, 0.01186438836157322, -0...",5823
21085,80879,https://simple.wikipedia.org/wiki/Reinickendorf,Reinickendorf,Reinickendorf is a borough of Berlin. It has a...,"[0.013133134692907333, 0.006712786387652159, -...","[0.01463380642235279, 0.005551774520426989, -0...",21085
2503,8017,https://simple.wikipedia.org/wiki/Dover%2C%20Kent,"Dover, Kent",Dover is also the name of the capital of Delaw...,"[-0.0040330663323402405, -0.001816988922655582...","[0.00325195025652647, -0.010359125211834908, 0...",2503


In [4]:
df.columns

Index(['id', 'url', 'title', 'text', 'title_vector', 'content_vector',
       'vector_id'],
      dtype='object')

**Parse Precomputed Embeddings**

The dataset already contains precomputed dense embeddings in the 'content_vector' column. We'll parse these from string to numpy arrays and use them directly for retrieval.

In [3]:
# take some time 3-5 mins
import ast

# Parse the 'content_vector' column from string to numpy array
content_vectors = df['content_vector'].apply(lambda x: np.array(ast.literal_eval(x), dtype=np.float32))

# Stack into a 2D numpy array
embeddings = np.stack(content_vectors.values)

print('Embeddings shape:', embeddings.shape)

Embeddings shape: (25000, 1536)


**Build a FAISS Index for Fast Similarity Search**

We will use FAISS to build an approximate nearest neighbor (ANN) index over the precomputed document embeddings. 

This allows for efficient retrieval of the most similar documents to a query embedding.

**How ANN Search Works in Practice**

1. Index Construction
- The dataset of embeddings (document vectors) is **preprocessed** into an **index**.  
- This index organizes vectors for fast similarity search.  
- Different libraries use different structures:  
  - **FAISS** → clustering & quantization.  
  - **Annoy** → random projection trees.  
  - **HNSW** → hierarchical navigable small-world graphs.  

2. Querying
- Given a **query embedding**, the algorithm searches the index.  
- Instead of scanning all vectors (brute-force), it looks only at a **subset likely to be close**.  
- Example: HNSW starts at an entry point in the graph and “walks” toward the nearest neighbors.  

3. Approximation
- ANN doesn’t guarantee the *exact* nearest neighbor.  
- But it finds **very close matches** much faster than brute-force.  
- ⚖️ Speed vs. accuracy can be tuned:  
  - Larger search → more accurate, slower.  
  - Smaller search → faster, slightly less accurate.  

4. Result
- Returns a list of the **most similar vectors (documents)** to the query.  
- Typically in **milliseconds**, even for millions of items.  
- These results feed into the **RAG pipeline** → (retrieval → context injection → generation).  


> **Note:**
> The document embeddings in this notebook were generated using OpenAI's "small" embedding model (e.g., `text-embedding-3-small`).
>
> - When running retrieval, always embed your queries using the **same OpenAI model** to ensure compatibility with the document vectors.
> - You do **not** need to use SentenceTransformers or any other embedding library for queries or documents in this workflow.
> - The FAISS index and search logic remain the same; just ensure all embeddings (documents and queries) come from the same OpenAI model.
> - For new queries, use the OpenAI API to get the embedding, then search the FAISS index as shown.

In [4]:
embeddings.shape[1]

1536

In [5]:
import faiss

# Dimension of embeddings
dim = embeddings.shape[1]

# Build a FAISS index (L2 similarity)
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

print(f"FAISS index contains {index.ntotal} vectors.")

FAISS index contains 25000 vectors.


**Define a Dense Retrieval Function**

This function will take a query string, encode it into an embedding (using the same model as used for the precomputed vectors), and use the FAISS index to retrieve the top-k most similar documents. 

If you have precomputed query embeddings, you can use those directly as well.

In [6]:
def dense_retrieve(query, model, index, df, top_k=5):
    
    # Encode the query using the same model as used for the document vectors
    query_emb = model.encode([query], convert_to_numpy=True).astype(np.float32)
    
    # Search FAISS index
    D, I = index.search(query_emb, top_k)
    
    # D: distances, I: indices
    results          = df.iloc[I[0]].copy()
    results['score'] = D[0]
    
    return results

**Run a Sample Query and Display Results**

Let's try a sample query and display the top retrieved documents, sorted by similarity (lower distance = more similar).

In [7]:
# Get the embedding for the query using OpenAI API
from openai import OpenAI
import numpy as np

In [8]:
# Example query
query = "What is the capital of France?"

# Initialize OpenAI client
client = OpenAI()  # Replace with your OpenAI API key

response = client.embeddings.create(
    input= query,
    model= "text-embedding-3-small"
)

query_emb = np.array(response.data[0].embedding, dtype=np.float32).reshape(1, -1)

# Search FAISS index
top_k = 5
D, I = index.search(query_emb, top_k)
results = df.iloc[I[0]].copy()
results['score'] = D[0]

# Display results
display_cols = ['id', 'title', 'score', 'text']
results[display_cols]

,id,title,score,text
19369,73701,North American F-86 Sabre,1.825309,"The F-86 Sabre (nicknamed the ""Sabre jet"") was..."
20447,78550,Republic P-47 Thunderbolt,1.847468,The P-47 Thunderbolt (also called The Jug ) wa...
19523,74703,General Dynamics F-16 Fighting Falcon,1.851971,The General Dynamics F-16 Fighting Falcon is a...
20147,77188,Grumman F4F Wildcat,1.853217,The F4F Wildcat was a fighter aircraft made by...
3616,10981,Copyleft,1.854619,Copyleft is a name for a type of a license for...


**Explanation: How Dense Retrieval Works**

- The query and documents are embedded into the same vector space using a neural model.
- Similarity is measured by distance (L2 or cosine) between vectors.
- FAISS enables fast nearest neighbor search over large embedding sets.
- Lower distance means higher similarity.

**Dos and Don'ts for Dense Retrieval:**

**Dos:**
- Use the same preprocessing for queries and documents (lowercasing, punctuation removal if needed).
- Use a model trained for semantic similarity (e.g., SentenceTransformer models).
- Normalize embeddings if using cosine similarity (not needed for L2).
- Batch encode for efficiency.

**Don'ts:**
- Don't use unrelated models (e.g., classification models) for embedding.
- Don't forget to check the embedding dimension matches the index.

**Hybrid Retrieval Tip:**
- Combine dense and sparse (BM25) retrieval for best results in many RAG applications.